INSTALLING DEPENDECIES AND LOADING THE DATA

In [3]:
pip install tensorflow pandas matplotlib scikit-learn

  Using cached tensorflow-2.20.0-cp313-cp313-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.5 kB)
  Using cached pandas-2.3.3-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (91 kB)
  Using cached matplotlib-3.10.7-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (11 kB)
  Using cached scikit_learn-1.7.2-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (11 kB)
  Using cached absl_py-2.3.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.9.23-py2.py3-none-any.whl.metadata (875 bytes)
  Using cached gast-0.6.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-manylinux2010_x86_64.whl.metadata (5.2 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached protobuf-6.33.0-cp39-abi3-manylinux2014_x86_64.whl.m

In [7]:
import os
import pandas as pd
import tensorflow as tf
import numpy as np

In [8]:
df = pd.read_csv(os.path.join('jigsaw-toxic-comment-classification-challenge','train.csv', 'train.csv'))

In [9]:
df.head(10)

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0
5,00025465d4725e87,"""\n\nCongratulations from me as well, use the ...",0,0,0,0,0,0
6,0002bcb3da6cb337,COCKSUCKER BEFORE YOU PISS AROUND ON MY WORK,1,1,1,0,1,0
7,00031b1e95af7921,Your vandalism to the Matt Shirvington article...,0,0,0,0,0,0
8,00037261f536c51d,Sorry if the word 'nonsense' was offensive to ...,0,0,0,0,0,0
9,00040093b2687caa,alignment on this subject and which are contra...,0,0,0,0,0,0


In [10]:
df.tail()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
159566,ffe987279560d7ff,""":::::And for the second time of asking, when ...",0,0,0,0,0,0
159567,ffea4adeee384e90,You should be ashamed of yourself \n\nThat is ...,0,0,0,0,0,0
159568,ffee36eab5c267c9,"Spitzer \n\nUmm, theres no actual article for ...",0,0,0,0,0,0
159569,fff125370e4aaaf3,And it looks like it was actually you who put ...,0,0,0,0,0,0
159570,fff46fc426af1f9a,"""\nAnd ... I really don't think you understand...",0,0,0,0,0,0


In [11]:
df.columns

Index(['id', 'comment_text', 'toxic', 'severe_toxic', 'obscene', 'threat',
       'insult', 'identity_hate'],
      dtype='object')

PREPROCESSING 

In [12]:
from tensorflow.keras.layers import TextVectorization

In [13]:
TextVectorization??

Init signature:
TextVectorization(
    max_tokens=None,
    standardize='lower_and_strip_punctuation',
    split='whitespace',
    ngrams=None,
    output_mode='int',
    output_sequence_length=None,
    pad_to_max_tokens=False,
    vocabulary=None,
    idf_weights=None,
    sparse=False,
    ragged=False,
    encoding='utf-8',
    name=None,
    **kwargs,
)
Source:        
@keras_export("keras.layers.TextVectorization")
class TextVectorization(Layer):
    """A preprocessing layer which maps text features to integer sequences.

    This layer has basic options for managing text in a Keras model. It
    transforms a batch of strings (one example = one string) into either a list
    of token indices (one example = 1D tensor of integer token indices) or a
    dense representation (one example = 1D tensor of float values representing
    data about the example's tokens). This layer is meant to handle natural
    language inputs. To handle simple string inputs (categorical strings or
    pr

In [14]:
X = df['comment_text']
y = df[df.columns[2:]].values

In [15]:
MAX_WORDS = 100000
#no of words in the vocab

In [16]:
vectorizer = TextVectorization(max_tokens=MAX_WORDS, output_sequence_length=400, output_mode='int')

In [17]:
vectorizer.adapt(X.values)

In [18]:
vectorized_text = vectorizer(X.values)

In [19]:
vectorized_text

<tf.Tensor: shape=(159571, 400), dtype=int64, numpy=
array([[  645,    76,     2, ...,     0,     0,     0],
       [    1,    54,  2489, ...,     0,     0,     0],
       [  425,   441,    70, ...,     0,     0,     0],
       ...,
       [32445,  7392,   383, ...,     0,     0,     0],
       [    5,    12,   534, ...,     0,     0,     0],
       [    5,     8,   130, ...,     0,     0,     0]],
      shape=(159571, 400))>

In [20]:
# CREATING A DATA PIPELINE MCSHBAP - map, cache, shuffle, batch, prefetch
dataset = tf.data.Dataset.from_tensor_slices((vectorized_text, y))
dataset = dataset.cache()
dataset = dataset.shuffle(160000)
dataset = dataset.batch(16)
dataset = dataset.prefetch(8) # helps prevent bottlenecks

In [21]:
bacth_X, batch_y = dataset.as_numpy_iterator().next() 
# X stores the vectorized text values and y stores the labels for toxicity

In [22]:
train = dataset.take(int(len(dataset)*.7))
val = dataset.skip(int(len(dataset)*.7)).take(int(len(dataset)*.2))
test = dataset.skip(int(len(dataset)*.9)).take(int(len(dataset)*.1))

CREATING THE SEQUENTIAL MODEL

In [23]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Bidirectional, Dense, Embedding

In [24]:
model = Sequential()
# Create the embedding layer 
model.add(Embedding(MAX_WORDS+1, 32))
# Bidirectional LSTM Layer
model.add(Bidirectional(LSTM(32, activation='tanh')))
# Feature extractor Fully connected layers
model.add(Dense(128, activation='relu'))
model.add(Dense(256, activation='relu'))
model.add(Dense(128, activation='relu'))
# Final layer 
model.add(Dense(6, activation='sigmoid'))

In [25]:
model.compile(loss='BinaryCrossentropy', optimizer='Adam')

In [27]:
from tensorflow.keras.callbacks import ModelCheckpoint

# Save only the best model (based on validation accuracy)
checkpoint = ModelCheckpoint(
    'best_model.h5',
    monitor='val_accuracy',   # you can also use 'val_loss'
    save_best_only=True,
    mode='max',
    verbose=1
)

history = model.fit(
    train,
    epochs=10,
    validation_data=val,
    callbacks=[checkpoint]
)


Epoch 1/10
6981/6981 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - loss: 0.0857

/home/jose/Desktop/ML Projects/comment_toxic/venv/lib/python3.13/site-packages/keras/src/callbacks/model_checkpoint.py:276: UserWarning: Can save best model only with val_accuracy available.
  if self._should_save_model(epoch, batch, logs, filepath):


6981/6981 ━━━━━━━━━━━━━━━━━━━━ 426s 61ms/step - loss: 0.0638 - val_loss: 0.0457
Epoch 2/10
6981/6981 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - loss: 0.0474

6981/6981 ━━━━━━━━━━━━━━━━━━━━ 431s 62ms/step - loss: 0.0467 - val_loss: 0.0445
Epoch 3/10
6980/6981 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - loss: 0.0423

6981/6981 ━━━━━━━━━━━━━━━━━━━━ 429s 61ms/step - loss: 0.0424 - val_loss: 0.0392
Epoch 4/10
6981/6981 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - loss: 0.0393

...

6981/6981 ━━━━━━━━━━━━━━━━━━━━ 432s 62ms/step - loss: 0.0315 - val_loss: 0.0273
Epoch 7/10
6981/6981 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - loss: 0.0285

6981/6981 ━━━━━━━━━━━━━━━━━━━━ 432s 62ms/step - loss: 0.0289 - val_loss: 0.0250
Epoch 8/10
6981/6981 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - loss: 0.0268

6981/6981 ━━━━━━━━━━━━━━━━━━━━ 429s 61ms/step - loss: 0.0266 - val_loss: 0.0224
Epoch 9/10
6981/6981 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - loss: 0.0229

6981/6981 ━━━━━━━━━━━━━━━━━━━━ 430s 62ms/step - loss: 0.0240 - val_loss: 0.0212
Epoch 10/10
6981/6981 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - loss: 0.0219

6981/6981 ━━━━━━━━━━━━━━━━━━━━ 431s 62ms/step - loss: 0.0223 - val_loss: 0.0188


In [26]:
import pickle

with open("vectorizer.pkl", "wb") as f:
    pickle.dump(vectorizer, f)

In [27]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [28]:
from matplotlib import pyplot as plt
plt.figure(figsize=(8,5))
pd.DataFrame(history.history).plot()
plt.show()

NameError: name 'history' is not defined

<Figure size 800x500 with 0 Axes>

MAKE PREDICTIONS

In [30]:
input_text = vectorizer('You freaking suck! I am going to kill you')


In [31]:
batch = test.as_numpy_iterator().next()

In [32]:
batch_X, batch_y = test.as_numpy_iterator().next()

In [33]:
batch_y

array([[0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [1, 0, 1, 0, 1, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0],
       [1, 0, 1, 0, 1, 0],
       [0, 0, 0, 0, 0, 0]])

In [34]:
res = model.predict(np.expand_dims(input_text,0))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 460ms/step


In [35]:
res

array([[0.5012914 , 0.5002184 , 0.49999583, 0.5009062 , 0.4994257 ,
        0.49809992]], dtype=float32)

In [36]:
(model.predict(batch_X) > 0.5).astype(int)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 187ms/step


array([[1, 1, 0, 1, 0, 0],
       [1, 1, 0, 1, 0, 0],
       [1, 1, 0, 1, 0, 0],
       [1, 1, 0, 1, 0, 0],
       [1, 1, 0, 1, 1, 0],
       [1, 1, 0, 1, 0, 0],
       [1, 1, 0, 1, 0, 0],
       [1, 1, 0, 1, 0, 0],
       [1, 0, 1, 1, 0, 0],
       [1, 1, 0, 1, 0, 0],
       [1, 1, 0, 1, 0, 0],
       [1, 1, 0, 1, 0, 0],
       [1, 1, 0, 1, 1, 0],
       [1, 1, 0, 1, 0, 0],
       [1, 1, 0, 1, 0, 0],
       [1, 1, 0, 1, 0, 0]])

EVALUATING THE MODEL

In [1]:
from tensorflow.keras.metrics import Precision, Recall, CategoricalAccuracy

2025-11-02 17:47:16.840133: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-02 17:47:16.845650: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-02 17:47:17.087627: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-02 17:47:18.234393: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off,

In [2]:
pre = Precision()
re = Recall()
acc = CategoricalAccuracy()

E0000 00:00:1762085841.736513   11674 cuda_executor.cc:1309] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1762085841.744014   11674 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [37]:
for batch in test.as_numpy_iterator():
    X_true, y_true = batch # Unpacking the batch
    yhat = model.predict(X_true) # Making the predictions

    y_true = y_true.flatten()
    yhat = yhat.flatten()

    pre.update_state(y_true, yhat)
    re.update_state(y_true, yhat)
    acc.update_state(y_true, yhat)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━

2025-11-02 17:53:47.541752: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [38]:
print(f'Precision: {pre.result().numpy()}, Recall:{re.result().numpy()}, Accuracy:{acc.result().numpy()}')

Precision: 0.0390268936753273, Recall:0.5721724033355713, Accuracy:0.004012036137282848


BUILDING THE GRADIO APP

In [41]:
pip install gradio jinja2

Note: you may need to restart the kernel to use updated packages.


In [4]:
import gradio as gr
import tensorflow as tf

2025-10-24 11:46:04.519620: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-10-24 11:46:04.524989: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-24 11:46:04.732810: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-24 11:46:05.629305: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off,

In [43]:
model.save('toxicity.h5')

In [39]:
model = tf.keras.models.load_model('best_model.h5')

In [40]:
input_str = vectorizer('hey i freaking hate you')

In [41]:
res = model.predict(np.expand_dims(input_str, 0))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 152ms/step


In [42]:
res

array([[0.7336658 , 0.04746709, 0.14353654, 0.14345038, 0.46096453,
        0.22956222]], dtype=float32)

In [6]:
def score_comment(comment):
    vectorized_comment = vectorizer([comment])
    results = model.predict(vectorized_comment)
    
    text = ''
    for idx, col in enumerate(df.columns[2:]):
        text += '{}: {}\n'.format(col, results[0][idx]>0.5)
    
    return text

In [7]:
interface = gr.Interface(
    fn=score_comment,
    inputs=gr.Textbox(lines=2, placeholder="Comment to score"),
    outputs=gr.Text()
)


In [8]:
interface.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://b97e17b2f5bc973151.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Traceback (most recent call last):
  File "/home/jose/Desktop/ML Projects/comment_toxic/venv/lib/python3.13/site-packages/gradio/queueing.py", line 759, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "/home/jose/Desktop/ML Projects/comment_toxic/venv/lib/python3.13/site-packages/gradio/route_utils.py", line 354, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<11 lines>...
    )
    ^
  File "/home/jose/Desktop/ML Projects/comment_toxic/venv/lib/python3.13/site-packages/gradio/blocks.py", line 2116, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<8 lines>...
    )
    ^
  File "/home/jose/Desktop/ML Projects/comment_toxic/venv/lib/python3.13/site-packages/gradio/blocks.py", line 1623, in call_function
    prediction = await anyio.to_thread.run_sync

In [43]:
import tensorflow as tf
import pickle

# Load model and vectorizer
model = tf.keras.models.load_model('best_model.h5')
with open('vectorizer.pkl', 'rb') as f:
    vectorizer = pickle.load(f)

# Assuming you already have a test dataset (e.g., created earlier)
# test = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(32)

# Define metrics
pre = tf.keras.metrics.Precision()
re = tf.keras.metrics.Recall()
acc = tf.keras.metrics.BinaryAccuracy()

# Evaluate in batches
for batch in test.as_numpy_iterator():
    X_true, y_true = batch
    yhat = model.predict(X_true)
    
    y_true = y_true.flatten()
    yhat = yhat.flatten()

    pre.update_state(y_true, yhat)
    re.update_state(y_true, yhat)
    acc.update_state(y_true, yhat)

# Print results
print(f"Precision: {pre.result().numpy():.4f}")
print(f"Recall: {re.result().numpy():.4f}")
print(f"Accuracy: {acc.result().numpy():.4f}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 150ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
1/1 ━━━━━━━

2025-11-02 17:56:56.165031: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


## 🎓 25PEC4CS1 Natural Language Processing — Full 6-Module Syllabus Pipeline Verification
The cell below executes the complete classical & statistical NLP pipeline (`toxicity_app/nlp_pipeline.py`) alongside the BiLSTM model, covering:
- **Module 1.4:** Conditional Probability, Bayes' Rule, Jigsaw Class Priors, Mean ($\mu$), Variance ($\sigma^2$), & Standard Deviation ($\sigma$)
- **Module 2:** Tokenization, Sentence Segmentation, Lemmatization, Porter Stemmer, Levenshtein Edit Distance (Adversarial De-obfuscation), PMI Collocations, and Laplace-Smoothed Bigram Language Model Perplexity
- **Module 3:** Penn Treebank POS Tagging using a Hidden Markov Model (HMM) + Exact Viterbi Decoder + Brill Transformation Rules
- **Module 4:** WordNet Synset Lexical Relations (Synonyms, Antonyms, Hypernyms, Hyponyms, Meronyms, Reversives), Lesk Word Sense Disambiguation (WSD), and TF-IDF Cosine Distance
- **Module 5:** Pragmatics & Discourse Processing — Hobbs' Algorithm & Centering Theory ($C_f, C_p, C_b$) Anaphora Resolution
- **Module 6:** Explainable AI (XAI) Token Attribution, Demographic Parity / Counterfactual Fairness Audit, and Regex PII Anonymization

In [ ]:
import pprint
from toxicity_app.nlp_pipeline import run_full_nlp_pipeline

sample_comment = "John is a st*pid id1ot. K1ll him! You are killing it on stage though. Contact: test@crce.edu.in"
report = run_full_nlp_pipeline(sample_comment)

print("=== 1. MULTI-LABEL TOXICITY PROBABILITIES & MODULE 1.4 STATS ===")
pprint.pprint(report["predictions"])
pprint.pprint(report["syllabus_analysis"]["module_1_preliminaries"]["probability_statistics"])

print("\n=== 2. MODULE 2: MORPHOLOGY, PORTER STEMMER, EDIT DISTANCE & BIGRAM PERPLEXITY ===")
pprint.pprint(report["syllabus_analysis"]["module_2_word_level"]["morphology"][:6])
pprint.pprint(report["syllabus_analysis"]["module_2_word_level"]["ngram_language_model"])

print("\n=== 3. MODULE 3: PENN TREEBANK HMM + VITERBI POS TAGGING ===")
print([(t["token"], t["pos_tag"]) for t in report["syllabus_analysis"]["module_3_syntax_level"]["tagged_tokens"]])

print("\n=== 4. MODULE 4: WORDNET LESK WSD & TF-IDF COSINE DISTANCE ===")
pprint.pprint(report["syllabus_analysis"]["module_4_semantic_analysis"]["wordnet_lesk_wsd"])
pprint.pprint(report["syllabus_analysis"]["module_4_semantic_analysis"]["tfidf_cosine_similarity"][:3])

print("\n=== 5. MODULE 5: HOBBS & CENTERING ANAPHORA RESOLUTION ===")
pprint.pprint(report["syllabus_analysis"]["module_5_pragmatics_discourse"]["anaphora_resolutions"])
pprint.pprint(report["syllabus_analysis"]["module_5_pragmatics_discourse"]["centering_table"])

print("\n=== 6. MODULE 6: EXPLAINABLE AI (XAI), FAIRNESS & PII REDACTION ===")
pprint.pprint(report["syllabus_analysis"]["module_6_ethics_and_xai"]["fairness_and_privacy"])
